# 03 — Bandpower Feature Extraction

## Purpose
Convert raw LFP voltage traces into time-binned frequency-domain bandpower
features used as input to the GRU-Transformer classifier.

## Input files (from notebook 01)
- `lfp_trials_{session_id}_probeC.npy`  — shape (N_trials, N_channels, N_samples)
- `stim_labels_{session_id}_probeC.npy` — shape (N_trials,)

## Output files (used by training scripts)
- `bp_time_features_rich532_{session_id}.npy` — shape (N_trials, K=10, D=532)
- `bp_time_labels_rich532_{session_id}.npy`   — shape (N_trials,)

## What this notebook does
1. **Welch PSD estimation** — estimates the power spectral density of each channel
2. **Bandpower integration** — integrates PSD over 7 frequency bands:
   delta (1–4 Hz), theta (4–8 Hz), alpha (8–13 Hz),
   beta1 (13–20 Hz), beta2 (20–30 Hz), gamma1 (30–60 Hz), gamma2 (60–120 Hz)
3. **Log scaling** — applies log10 to stabilise the numerical range
4. **Label collapsing** — merges stimulus sub-variants into canonical categories
5. **Time binning** — divides each 1-second trial into K=10 equal windows;
   bandpower is computed per window to expose temporal dynamics to the model
6. **Dimension alignment** — crops all sessions to the common feature dimension
   D=532 (minimum across sessions) to enable cross-session concatenation

## ⚠️ Note on paths
Update `data_dir` in the cells below to point to your local data directory.

In [2]:
import os
import numpy as np
import pandas as pd
from scipy.signal import welch

In [6]:
data_dir = "/projectnb/cs523aw/students/pangelos/RetreivedData"
session_ids = [
    766640955,
    767871931,
    768515987,
    # add more later
]

bands = {
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta":  (12, 30),
    "gamma": (30, 80),
}

fs = 1250.0  # Hz

def compute_bandpower(trial, fs, bands):
    C, T = trial.shape
    freqs, psd = welch(trial, fs=fs, nperseg=min(256, T), axis=1)  # (C, F)
    feats = []
    for (fmin, fmax) in bands.values():
        mask = (freqs >= fmin) & (freqs <= fmax)
        bp = psd[:, mask].mean(axis=1)  # (C,)
        feats.append(bp)
    feats = np.stack(feats, axis=1)  # (C, n_bands)
    return np.log10(feats + 1e-12)

collapse_map = {
    "drifting_gratings_75_repeats": "drifting_gratings",
    "drifting_gratings_contrast": "drifting_gratings",
    "natural_movie_one_more_repeats": "natural_movie",
    "natural_movie_one_shuffled": "natural_movie",
}

def collapse_label(lbl):
    return collapse_map.get(lbl, lbl)

for sid in session_ids:
    trials_path = os.path.join(data_dir, f"lfp_trials_{sid}_probeC.npy")
    labels_path = os.path.join(data_dir, f"stim_labels_{sid}_probeC.npy")
    if not (os.path.exists(trials_path) and os.path.exists(labels_path)):
        print(f"Missing files for session {sid}, skipping.")
        continue

    print(f"Processing session {sid}...")
    X = np.load(trials_path, mmap_mode="r")  # (N, C, T)
    y_raw = np.load(labels_path, allow_pickle=True)
    y_collapsed = np.array([collapse_label(lbl) for lbl in y_raw])

    all_features = []
    all_labels = []

    n_trials, C, T = X.shape
    for i in range(n_trials):
        trial = X[i]  # (C, T)
        if np.isnan(trial).any():
            continue
        feats = compute_bandpower(trial, fs, bands)  # (C, n_bands)
        feat_vec = feats.reshape(-1)  # (C * n_bands) for THIS session
        all_features.append(feat_vec)
        all_labels.append(y_collapsed[i])

    all_features = np.stack(all_features)
    all_labels = np.array(all_labels, dtype=object)

    print(f"Session {sid}: feature shape {all_features.shape}")
    print("Label counts:\n", pd.Series(all_labels).value_counts())

    np.save(os.path.join(data_dir, f"bp_features_pc_{sid}.npy"), all_features.astype("float32"))
    np.save(os.path.join(data_dir, f"bp_labels_pc_{sid}.npy"), all_labels)


Processing session 766640955...
Session 766640955: feature shape (13785, 328)
Label counts:
 natural_movie        6000
spontaneous          3000
gabors               3000
drifting_gratings    1140
dot_motion            495
flashes               150
dtype: int64
Processing session 767871931...
Session 767871931: feature shape (13785, 324)
Label counts:
 natural_movie        6000
spontaneous          3000
gabors               3000
drifting_gratings    1140
dot_motion            495
flashes               150
dtype: int64
Processing session 768515987...
Session 768515987: feature shape (13710, 324)
Label counts:
 natural_movie        6000
gabors               3000
spontaneous          3000
drifting_gratings    1140
dot_motion            420
flashes               150
dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

data_dir = "/projectnb/cs523aw/students/pangelos/RetreivedData"

sessions_324 = [767871931, 768515987]

Xs = []
ys = []
for sid in sessions_324:
    Xs.append(np.load(os.path.join(data_dir, f"bp_features_pc_{sid}.npy")))
    ys.append(np.load(os.path.join(data_dir, f"bp_labels_pc_{sid}.npy"), allow_pickle=True))

X = np.concatenate(Xs, axis=0)
y = np.concatenate(ys, axis=0)
print("Combined shape:", X.shape)
print("Label counts:\n", pd.Series(y).value_counts())

keep_classes = ["drifting_gratings", "natural_movie", "spontaneous"]
mask = np.isin(y, keep_classes)
X_3 = X[mask]
y_3 = y[mask]
print("3-way counts:\n", pd.Series(y_3).value_counts())

labels_series = pd.Series(y_3)
class_counts = labels_series.value_counts()
min_n = class_counts.min()
print("Balancing to", min_n, "per class.")

rng = np.random.default_rng(seed=0)
keep_indices = []
for label, idxs in labels_series.groupby(labels_series).groups.items():
    idxs = np.array(list(idxs))
    if len(idxs) > min_n:
        idxs = rng.choice(idxs, size=min_n, replace=False)
    keep_indices.append(idxs)

keep_indices = np.concatenate(keep_indices)
rng.shuffle(keep_indices)

X_bal = X_3[keep_indices]
y_bal = y_3[keep_indices]
print("Balanced:", X_bal.shape)
print("Balanced counts:\n", pd.Series(y_bal).value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.3, stratify=y_bal, random_state=0
)

print("Train:", X_train.shape, "Test:", X_test.shape)

y_train_enc, classes = pd.factorize(y_train)
label_to_int = {label: i for i, label in enumerate(classes)}
y_test_enc = pd.Series(y_test).map(label_to_int).values

np.save(os.path.join(data_dir, "bp_X_train_3way_324.npy"), X_train.astype("float32"))
np.save(os.path.join(data_dir, "bp_y_train_3way_324.npy"), y_train_enc.astype("int64"))
np.save(os.path.join(data_dir, "bp_X_test_3way_324.npy"), X_test.astype("float32"))
np.save(os.path.join(data_dir, "bp_y_test_3way_324.npy"), y_test_enc.astype("int64"))
np.save(os.path.join(data_dir, "bp_classes_3way_324.npy"), classes)


Combined shape: (27495, 324)
Label counts:
 natural_movie        12000
spontaneous           6000
gabors                6000
drifting_gratings     2280
dot_motion             915
flashes                300
dtype: int64
3-way counts:
 natural_movie        12000
spontaneous           6000
drifting_gratings     2280
dtype: int64
Balancing to 2280 per class.
Balanced: (6840, 324)
Balanced counts:
 spontaneous          2280
drifting_gratings    2280
natural_movie        2280
dtype: int64
Train: (4788, 324) Test: (2052, 324)


In [10]:
data_dir = "/projectnb/cs523aw/students/pangelos/RetreivedData"

# Sessions with 324-dim per-channel features
session_ids = [767871931, 768515987]

# Frequency bands (Hz)
bands = {
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta":  (12, 30),
    "gamma": (30, 80),
}
band_list = list(bands.values())

fs = 1250.0  # LFP sampling rate (Hz)
K  = 10      # number of time bins per 1 s trial

def bandpower_window(trial_seg, fs, band_list):
    """
    trial_seg: (C, T_seg)
    returns: (C, n_bands) bandpower in log10 units, finite.
    """
    C, T = trial_seg.shape
    if T < 4:
        # too short to estimate PSD; return small constant
        return np.full((C, len(band_list)), -12.0, dtype=np.float32)

    freqs, psd = welch(trial_seg, fs=fs, nperseg=min(128, T), axis=1)
    feats = []
    for (fmin, fmax) in band_list:
        mask = (freqs >= fmin) & (freqs <= fmax)
        if not np.any(mask):
            bp = np.zeros(C, dtype=np.float32)
        else:
            bp = psd[:, mask].mean(axis=1).astype(np.float32)
        bp = np.clip(bp, 1e-12, None)  # avoid log10(0)
        feats.append(bp)
    feats = np.stack(feats, axis=1)  # (C, n_bands)
    return np.log10(feats).astype(np.float32)

collapse_map = {
    "drifting_gratings_75_repeats": "drifting_gratings",
    "drifting_gratings_contrast": "drifting_gratings",
    "natural_movie_one_more_repeats": "natural_movie",
    "natural_movie_one_shuffled": "natural_movie",
}

def collapse_label(lbl):
    return collapse_map.get(lbl, lbl)

for sid in session_ids:
    trials_path = os.path.join(data_dir, f"lfp_trials_{sid}_probeC.npy")
    labels_path = os.path.join(data_dir, f"stim_labels_{sid}_probeC.npy")
    if not (os.path.exists(trials_path) and os.path.exists(labels_path)):
        print(f"Missing files for session {sid}, skipping.")
        continue

    print(f"Processing session {sid} for time-binned bandpower...")
    X = np.load(trials_path, mmap_mode="r")  # (N, C, T)
    y_raw = np.load(labels_path, allow_pickle=True)
    y_collapsed = np.array([collapse_label(lbl) for lbl in y_raw])

    N, C, T = X.shape
    seg_len = T // K
    print("Trial shape:", X.shape, "seg_len:", seg_len)

    all_feats = []
    all_labels = []

    dropped_trials = 0

    for i in range(N):
        trial = X[i]  # (C, T)
        if np.isnan(trial).any() or np.isinf(trial).any():
            dropped_trials += 1
            continue

        seg_feats = []
        for k in range(K):
            start = k * seg_len
            end   = (k + 1) * seg_len if k < K - 1 else T
            seg = trial[:, start:end]          # (C, T_seg)
            bp  = bandpower_window(seg, fs, band_list)  # (C, n_bands)
            seg_feats.append(bp.reshape(-1))   # flatten C*n_bands

        seg_feats = np.stack(seg_feats, axis=0)  # (K, C*n_bands)

        if np.isnan(seg_feats).any() or np.isinf(seg_feats).any():
            dropped_trials += 1
            continue

        all_feats.append(seg_feats)
        all_labels.append(y_collapsed[i])

    all_feats = np.stack(all_feats, axis=0)   # (N_eff, K, C*n_bands)
    all_labels = np.array(all_labels, dtype=object)

    print(f"Session {sid}: time-binned feature shape {all_feats.shape}")
    print("Label counts:\n", pd.Series(all_labels).value_counts())
    print(f"Dropped trials due to NaN/inf or bad input: {dropped_trials}")

    out_feat = os.path.join(data_dir, f"bp_time_features_{sid}.npy")
    out_lab  = os.path.join(data_dir, f"bp_time_labels_{sid}.npy")
    np.save(out_feat, all_feats.astype("float32"))
    np.save(out_lab, all_labels)
    print(f"Saved {out_feat} and {out_lab}")


Processing session 767871931 for time-binned bandpower...
Trial shape: (13785, 81, 1249) seg_len: 124
Session 767871931: time-binned feature shape (13785, 10, 324)
Label counts:
 natural_movie        6000
spontaneous          3000
gabors               3000
drifting_gratings    1140
dot_motion            495
flashes               150
dtype: int64
Dropped trials due to NaN/inf or bad input: 0
Saved /projectnb/cs523aw/students/pangelos/RetreivedData/bp_time_features_767871931.npy and /projectnb/cs523aw/students/pangelos/RetreivedData/bp_time_labels_767871931.npy
Processing session 768515987 for time-binned bandpower...
Trial shape: (13710, 81, 1249) seg_len: 124
Session 768515987: time-binned feature shape (13710, 10, 324)
Label counts:
 natural_movie        6000
gabors               3000
spontaneous          3000
drifting_gratings    1140
dot_motion            420
flashes               150
dtype: int64
Dropped trials due to NaN/inf or bad input: 0
Saved /projectnb/cs523aw/students/pangel

In [3]:
data_dir = "/projectnb/cs523aw/students/pangelos/RetreivedData"

session_ids = [767871931, 768515987, 779839471, 778998620, 778240327, 771990200, 771160300 ]

# Richer band set
bands = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta1": (12, 20),
    "beta2": (20, 30),
    "gamma1": (30, 55),
    "gamma2": (55, 90),
}
band_list = list(bands.values())

fs = 1250.0
K  = 10      # time bins per trial

def bandpower_window(trial_seg, fs, band_list):
    C, T = trial_seg.shape
    if T < 4:
        return np.full((C, len(band_list)), -12.0, dtype=np.float32)
    freqs, psd = welch(trial_seg, fs=fs, nperseg=min(128, T), axis=1)
    feats = []
    for (fmin, fmax) in band_list:
        mask = (freqs >= fmin) & (freqs <= fmax)
        if not np.any(mask):
            bp = np.zeros(C, dtype=np.float32)
        else:
            bp = psd[:, mask].mean(axis=1).astype(np.float32)
        bp = np.clip(bp, 1e-12, None)
        feats.append(bp)
    feats = np.stack(feats, axis=1)  # (C, n_bands)
    return np.log10(feats).astype(np.float32)

# No collapsing: use all original labels
for sid in session_ids:
    trials_path = os.path.join(data_dir, f"lfp_trials_{sid}_probeC.npy")
    labels_path = os.path.join(data_dir, f"stim_labels_{sid}_probeC.npy")
    if not (os.path.exists(trials_path) and os.path.exists(labels_path)):
        print(f"Missing files for session {sid}, skipping.")
        continue

    print(f"Processing session {sid} for time-binned bandpower (rich bands)...")
    X = np.load(trials_path, mmap_mode="r")  # (N, C, T)
    y = np.load(labels_path, allow_pickle=True)

    N, C, T = X.shape
    seg_len = T // K
    print("Trial shape:", X.shape, "seg_len:", seg_len)

    all_feats = []
    all_labels = []
    dropped_trials = 0

    for i in range(N):
        trial = X[i]
        if np.isnan(trial).any() or np.isinf(trial).any():
            dropped_trials += 1
            continue

        seg_feats = []
        for k in range(K):
            start = k * seg_len
            end   = (k + 1) * seg_len if k < K - 1 else T
            seg = trial[:, start:end]
            bp  = bandpower_window(seg, fs, band_list)  # (C, n_bands)
            seg_feats.append(bp.reshape(-1))            # (C * n_bands,)

        seg_feats = np.stack(seg_feats, axis=0)  # (K, C * n_bands)
        if np.isnan(seg_feats).any() or np.isinf(seg_feats).any():
            dropped_trials += 1
            continue

        all_feats.append(seg_feats)
        all_labels.append(y[i])

    all_feats = np.stack(all_feats, axis=0)   # (N_eff, K, D)
    all_labels = np.array(all_labels, dtype=object)

    print(f"Session {sid}: time-binned feature shape {all_feats.shape}")
    print("Label counts:\n", pd.Series(all_labels).value_counts())
    print(f"Dropped trials: {dropped_trials}")

    out_feat = os.path.join(data_dir, f"bp_time_features_rich_{sid}.npy")
    out_lab  = os.path.join(data_dir, f"bp_time_labels_rich_{sid}.npy")
    np.save(out_feat, all_feats.astype("float32"))
    np.save(out_lab, all_labels)
    print(f"Saved {out_feat} and {out_lab}")


Processing session 767871931 for time-binned bandpower (rich bands)...
Trial shape: (13785, 81, 1249) seg_len: 124
Session 767871931: time-binned feature shape (13785, 10, 567)
Label counts:
 natural_movie_one_shuffled        3000
spontaneous                       3000
gabors                            3000
natural_movie_one_more_repeats    3000
drifting_gratings_75_repeats       600
drifting_gratings_contrast         540
dot_motion                         495
flashes                            150
dtype: int64
Dropped trials: 0
Saved /projectnb/cs523aw/students/pangelos/RetreivedData/bp_time_features_rich_767871931.npy and /projectnb/cs523aw/students/pangelos/RetreivedData/bp_time_labels_rich_767871931.npy
Processing session 768515987 for time-binned bandpower (rich bands)...
Trial shape: (13710, 81, 1249) seg_len: 124
Session 768515987: time-binned feature shape (13710, 10, 567)
Label counts:
 gabors                            3000
spontaneous                       3000
natural_movie

In [6]:
data_dir = "/projectnb/cs523aw/students/pangelos/RetreivedData"

sessions = [767871931, 768515987,
            779839471, 778998620,
            778240327, 771990200, 771160300]

common_D = 532

for sid in sessions:
    feat_path = os.path.join(data_dir, f"bp_time_features_rich_{sid}.npy")
    lab_path  = os.path.join(data_dir, f"bp_time_labels_rich_{sid}.npy")

    X = np.load(feat_path)  # (N, K, D_s)
    y = np.load(lab_path, allow_pickle=True)

    N, K, D_s = X.shape
    print(f"Session {sid}: original shape {X.shape}")

    if D_s < common_D:
        print(f"  Skipping {sid}: D={D_s} < {common_D}")
        continue

    X_crop = X[:, :, :common_D]  # simple crop on last axis
    print(f"  Cropped shape: {X_crop.shape}")
    print("  Label counts:\n", pd.Series(y).value_counts())

    out_feat = os.path.join(data_dir, f"bp_time_features_rich{common_D}_{sid}.npy")
    out_lab  = os.path.join(data_dir, f"bp_time_labels_rich{common_D}_{sid}.npy")

    np.save(out_feat, X_crop)
    np.save(out_lab, y)
    print("  Saved", out_feat, "and", out_lab)



Session 767871931: original shape (13785, 10, 567)
  Cropped shape: (13785, 10, 532)
  Label counts:
 natural_movie_one_shuffled        3000
spontaneous                       3000
gabors                            3000
natural_movie_one_more_repeats    3000
drifting_gratings_75_repeats       600
drifting_gratings_contrast         540
dot_motion                         495
flashes                            150
dtype: int64
  Saved /projectnb/cs523aw/students/pangelos/RetreivedData/bp_time_features_rich532_767871931.npy and /projectnb/cs523aw/students/pangelos/RetreivedData/bp_time_labels_rich532_767871931.npy
Session 768515987: original shape (13710, 10, 567)
  Cropped shape: (13710, 10, 532)
  Label counts:
 gabors                            3000
spontaneous                       3000
natural_movie_one_more_repeats    3000
natural_movie_one_shuffled        3000
drifting_gratings_75_repeats       600
drifting_gratings_contrast         540
dot_motion                         420
flashes 